[Open in Colab](https://colab.research.google.com/github/BoydVonstocker/Introduction-to-Quantum-Computing/blob/main/Notebks/06%20-%20Foundational_Algorithms.ipynb)


# Notebook 06 — Foundational Quantum Algorithms

This notebook is the algorithms workshop. Every circuit below uses the same pattern you already know: prepare a superposition, query an oracle once (or a few times), interfere, measure.

| Algorithm | Question it answers | Cost |
|---|---|---|
| Deutsch | is 1-bit $f$ constant or balanced? | 1 query instead of 2 |
| Deutsch–Jozsa | is $n$-bit $f$ constant or balanced? | 1 query instead of $2^{n-1}+1$ |
| Bernstein–Vazirani | recover secret $s$ from $f(x)=s\cdot x$ | 1 query instead of $n$ |
| Grover | find a marked item among $N$ | $\sim\sqrt{N}$ queries instead of $\sim N$ |
| Teleportation | move an unknown qubit with entanglement + 2 bits | protocol, not a speed-up |

**This is a new notebook, remember to bring your tools.**

Bitstring convention: Qiskit prints qubit 0 on the **right**. A measured label `01` means qubit 1 is `0` and qubit 0 is `1`.


In [ ]:
!pip install qiskit qiskit-aer pylatexenc -q


In [ ]:
from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

sim = AerSimulator()


def show(qc, shots=1024, title="counts"):
    display(qc.draw("mpl"))
    counts = sim.run(qc, shots=shots).result().get_counts()
    print(title, counts)
    display(plot_histogram(counts))
    return counts


# 1. Deutsch's algorithm

**Problem.** You are given an oracle for

$$
f:\{0,1\}\to\{0,1\}
$$

promised to be either **constant** ($f(0)=f(1)$) or **balanced** ($f(0)\neq f(1)$). Classically you must evaluate $f$ twice. Deutsch decides the promise with **one** coherent query.


## The four possible functions

| name | $f(0)$ | $f(1)$ | type |
|---|---|---|---|
| `const0` | 0 | 0 | constant |
| `const1` | 1 | 1 | constant |
| `identity` | 0 | 1 | balanced |
| `NOT` | 1 | 0 | balanced |

The two-qubit *bit oracle* implements

$$
U_f\,|x\rangle|y\rangle = |x\rangle|y\oplus f(x)\rangle.
$$

Prepare the target in $|-\rangle = HX|0\rangle$ and this becomes phase kickback on the data qubit:

$$
U_f\,|x\rangle|-\rangle = (-1)^{f(x)}\,|x\rangle|-\rangle.
$$


In [ ]:
def deutsch_oracle(kind: str) -> QuantumCircuit:
    """Two-qubit oracle: |x>|y> -> |x>|y XOR f(x)>."""
    o = QuantumCircuit(2, name=f"Uf[{kind}]")
    if kind == "const0":
        pass
    elif kind == "const1":
        o.x(1)
    elif kind == "identity":
        o.cx(0, 1)
    elif kind == "NOT":
        o.x(0)
        o.cx(0, 1)
        o.x(0)
    else:
        raise ValueError(kind)
    return o


for kind in ("const0", "const1", "identity", "NOT"):
    print(kind)
    print(deutsch_oracle(kind).draw("text"))


## The Deutsch circuit

1. Data qubit starts in $|0\rangle$, target in $|1\rangle$.
2. Hadamard both (data $\to|+\rangle$, target $\to|-\rangle$).
3. Query $U_f$ once.
4. Hadamard the data qubit and measure it.

- measured `0` $\Rightarrow$ **constant**
- measured `1` $\Rightarrow$ **balanced**

After the oracle the data qubit is $|+\rangle$ if $f(0)=f(1)$ and $|-\rangle$ if $f(0)\neq f(1)$. The final $H$ maps those onto $|0\rangle$ and $|1\rangle$.


In [ ]:
def deutsch(kind: str) -> QuantumCircuit:
    qc = QuantumCircuit(2, 1)
    qc.x(1)
    qc.h(0)
    qc.h(1)
    qc.barrier(label="oracle")
    qc.compose(deutsch_oracle(kind), inplace=True)
    qc.barrier()
    qc.h(0)
    qc.measure(0, 0)
    return qc


show(deutsch("identity"), title="balanced / identity")
print("---")
for kind in ("const0", "const1", "identity", "NOT"):
    counts = sim.run(deutsch(kind), shots=1024).result().get_counts()
    print(f"{kind:10s}  {counts}")


# 2. Deutsch–Jozsa

Replace the single data qubit by $n$ qubits. The promise is now: $f:\{0,1\}^n\to\{0,1\}$ is either **constant** or **balanced** (exactly half of the inputs return 1).

The circuit is the same sandwich

$$
H^{\otimes n}\;U_f\;H^{\otimes n}
$$

on the data register, with the target still in $|-\rangle$.

- all-zero bitstring $\Rightarrow$ constant
- any other bitstring $\Rightarrow$ balanced

Deutsch is the $n=1$ case. Below we run $n=2$.


In [ ]:
def dj_oracle(n: int, kind: str) -> QuantumCircuit:
    """n data qubits + 1 target. kind in {const0, const1, x0, xor}."""
    o = QuantumCircuit(n + 1, name=f"DJ[{kind}]")
    if kind == "const0":
        pass
    elif kind == "const1":
        o.x(n)
    elif kind == "x0":
        o.cx(0, n)            # balanced: f = x_0
    elif kind == "xor":
        for i in range(n):    # balanced: f = x_0 XOR ... XOR x_{n-1}
            o.cx(i, n)
    else:
        raise ValueError(kind)
    return o


def deutsch_jozsa(n: int, kind: str) -> QuantumCircuit:
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(range(n + 1))
    qc.barrier()
    qc.compose(dj_oracle(n, kind), inplace=True)
    qc.barrier()
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc


for kind in ("const0", "const1", "x0", "xor"):
    counts = sim.run(deutsch_jozsa(2, kind), shots=1024).result().get_counts()
    print(f"{kind:8s}  {counts}")

show(deutsch_jozsa(2, "xor"), title="DJ n=2, f = x0 XOR x1 (balanced)")


For the balanced oracle `x0` the peak sits on the Qiskit label `01` (a `1` on qubit 0, the right-hand bit), not on a textbook left-endian `10`.


# 3. Bernstein–Vazirani

**Problem.** There is a secret bitstring $s\in\{0,1\}^n$ and an oracle for

$$
f(x) = s\cdot x \pmod{2} = s_0 x_0 \oplus \cdots \oplus s_{n-1}x_{n-1}.
$$

Classically you learn $s$ one bit at a time ($n$ queries). Bernstein–Vazirani learns the whole string with **one** query.

The circuit is Deutsch–Jozsa with a different promise: $f$ is linear. Measuring the data register returns $s$ itself.

Write $s$ the way Qiskit will print it: **qubit 0 on the right**. So `secret = "101"` means $s_2 s_1 s_0 = 101$.


In [ ]:
def bv_oracle(secret: str) -> QuantumCircuit:
    """secret is a Qiskit-style label: qubit 0 is the RIGHTMOST character."""
    n = len(secret)
    o = QuantumCircuit(n + 1, name=f"BV[{secret}]")
    bits = secret[::-1]          # bits[i] multiplies qubit i
    for i, bit in enumerate(bits):
        if bit == "1":
            o.cx(i, n)
    return o


def bernstein_vazirani(secret: str) -> QuantumCircuit:
    n = len(secret)
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(range(n + 1))
    qc.barrier()
    qc.compose(bv_oracle(secret), inplace=True)
    qc.barrier()
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc


secret = "101"
show(bernstein_vazirani(secret), title=f"recovered s (want {secret})")

for secret in ("001", "010", "111", "100"):
    counts = sim.run(bernstein_vazirani(secret), shots=512).result().get_counts()
    print(f"secret={secret}  measured={counts}")


Why it works: with the target in $|-\rangle$,

$$
U_f|x\rangle|-\rangle = (-1)^{s\cdot x}|x\rangle|-\rangle.
$$

A layer of $H^{\otimes n}$ on the uniform superposition turns that phase pattern into the computational-basis state $|s\rangle$.


# 4. Grover's search

**Problem.** Among $N=2^n$ basis states, one item is marked by an oracle that flips its phase:

$$
O|x\rangle = \begin{cases}-|x\rangle & x\text{ marked}\\ |x\rangle & \text{otherwise.}\end{cases}
$$

Grover rotates the state toward the marked item. For one marked item the optimal iteration count is about

$$
\frac{\pi}{4}\sqrt{N}.
$$

For $n=2$ ($N=4$) that number is **1**. One Grover iteration finds the marked state with certainty on an ideal simulator.


## One iteration = oracle + diffuser

The **diffuser** reflects about the uniform superposition $|s\rangle = H^{\otimes n}|0\rangle$:

$$
D = 2|s\rangle\langle s| - I = H^{\otimes n}\,(2|0\rangle\langle 0|-I)\,H^{\otimes n}.
$$

On two qubits, $2|00\rangle\langle00|-I$ is a phase flip of every state except $|00\rangle$, which is $X\otimes X$ then $CZ$ then $X\otimes X$ (up to a global phase).


In [ ]:
def grover_oracle(marked: str) -> QuantumCircuit:
    """Phase-flip the marked label. marked is a Qiskit bitstring (qubit 0 on the right)."""
    o = QuantumCircuit(2, name=f"O[{marked}]")
    bits = marked[::-1]          # bits[i] is the desired value of qubit i
    for i, bit in enumerate(bits):
        if bit == "0":
            o.x(i)
    o.cz(0, 1)                   # phase-flip |11> in the flipped frame
    for i, bit in enumerate(bits):
        if bit == "0":
            o.x(i)
    return o


def diffuser() -> QuantumCircuit:
    d = QuantumCircuit(2, name="D")
    d.h([0, 1])
    d.x([0, 1])
    d.cz(0, 1)
    d.x([0, 1])
    d.h([0, 1])
    return d


def grover(marked: str, iterations: int = 1) -> QuantumCircuit:
    qc = QuantumCircuit(2, 2)
    qc.h([0, 1])
    for _ in range(iterations):
        qc.barrier(label="oracle")
        qc.compose(grover_oracle(marked), inplace=True)
        qc.barrier(label="diffuser")
        qc.compose(diffuser(), inplace=True)
    qc.measure([0, 1], [0, 1])
    return qc


show(grover("11", iterations=1), title="Grover n=2, marked=11")

for marked in ("00", "01", "10", "11"):
    counts = sim.run(grover(marked, 1), shots=1024).result().get_counts()
    print(f"marked={marked}  {counts}")


## Why one iteration is enough for $N=4$

Start at the uniform state, an angle $\arcsin(1/2)=30^\circ$ above the unmarked plane. Each Grover step adds $2\times 30^\circ=60^\circ$. After one step the state sits on the marked axis.

Zero iterations is a uniform guess. Two iterations overshoot back toward uniform.


In [ ]:
marked = "10"
for k in (0, 1, 2):
    counts = sim.run(grover(marked, k), shots=1024).result().get_counts()
    print(f"iterations={k}  {counts}")


# 5. Quantum teleportation

Teleportation is a *protocol*, not a query speed-up. It moves an unknown qubit $|\psi\rangle$ from Alice to Bob using one shared Bell pair and two classical bits. The original qubit is measured, so no cloning happens.

Outline:

1. Alice and Bob share $|\Phi^+\rangle$ on qubits $A$ and $B$.
2. Alice holds the message $M$ in some state $|\psi\rangle$.
3. Alice applies CNOT$(M\to A)$ then $H$ on $M$.
4. Those two qubits control corrections on Bob: $X$ from $A$, $Z$ from $M$.
5. Bob's qubit is $|\psi\rangle$.

The circuit below uses *deferred measurement*: the corrections are ordinary CNOT / CZ from Alice's wires. That runs on any Aer simulator.


In [ ]:
def teleport(prepare_message):
    """prepare_message(qc, q) writes |psi> onto qubit q.
    Qubit 0 = message, 1 = Alice, 2 = Bob. Measure Bob only.
    """
    qc = QuantumCircuit(3, 1)
    prepare_message(qc, 0)
    qc.barrier(label="msg")
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier(label="bell")
    qc.cx(0, 1)
    qc.h(0)
    qc.barrier(label="correct")
    qc.cx(1, 2)      # X correction
    qc.cz(0, 2)      # Z correction
    qc.measure(2, 0)
    return qc


def prep_one(qc, q):
    qc.x(q)


def prep_plus(qc, q):
    qc.h(q)


def prep_minus(qc, q):
    qc.x(q)
    qc.h(q)


show(teleport(prep_one), title="teleport |1>: Bob should always be 1")
show(teleport(prep_plus), title="teleport |+>: Bob is 50/50 in the Z basis")


To see that the teleported $|+\rangle$ really arrived, rotate Bob into the $X$ basis before measuring: an extra $H$ on qubit 2 should turn the histogram into a sharp `0`.


In [ ]:
qc = QuantumCircuit(3, 1)
prep_plus(qc, 0)
qc.h(1)
qc.cx(1, 2)
qc.cx(0, 1)
qc.h(0)
qc.cx(1, 2)
qc.cz(0, 2)
qc.h(2)            # measure Bob in the X basis
qc.measure(2, 0)
show(qc, title="teleport |+>, measure Bob in X (want 0)")


# What to notice across all five

They share one skeleton:

1. **Hadamards** create a superposition of every input the oracle might see.
2. The **oracle** writes a phase $(-1)^{f(x)}$ — kickback from a $|-\rangle$ target, or a direct phase gate.
3. Another layer of **Hadamards** (or the Grover diffuser) turns that phase pattern into a computational-basis peak.
4. **Measure.** For Deutsch / DJ / BV that is the whole algorithm. Grover repeats steps 2–3. Teleportation swaps the oracle for a Bell pair and two correction bits.

That skeleton is also the start of Simon, quantum Fourier sampling, and Shor: interfere a periodic phase, then read the period.


## Exercises

**Deutsch / Deutsch–Jozsa**
1. Confirm by hand that $U_f|x\rangle|-\rangle = (-1)^{f(x)}|x\rangle|-\rangle$ for the `identity` oracle.
2. What happens in Deutsch if you leave the target in $|0\rangle$ instead of $|-\rangle$?
3. Build DJ for $n=3$ with balanced oracle $f(x)=x_0\oplus x_2$ and check you never measure `000`.

**Bernstein–Vazirani**
4. Recover $s=1101$ (Qiskit label, qubit 0 on the right).
5. Set the secret to `0000`. Which algorithm have you just rerun?

**Grover**
6. Mark `01` and compare 0, 1, and 2 iterations.
7. Phase-flip both `01` and `10`. With two marked items in $N=4$, how many iterations look best?

**Teleportation**
8. Teleport $|-\rangle$ and measure Bob in the $X$ basis (add $H$ on qubit 2 before `measure`). You should see a deterministic `1`.


In [ ]:
# Exercise workspace
